In [453]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()
base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
# model = "gemini-3-flash"
model = "claude-opus-4-5-thinking"

In [454]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 8000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [455]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["x.com","infoq.cn","aibase.com"],
}

In [456]:
messages = []

# 从 web_search_schema 中获取允许的域名，构建搜索限定提示
allowed_domains = web_search_schema.get('allowed_domains', None)
domain_hint = ""
if allowed_domains:
    sites = " OR ".join([f"site:{d}" for d in allowed_domains])
    domain_hint = f"\n    请仅从以下网站搜索和引用内容：{', '.join(allowed_domains)}（搜索时请使用 {sites}）。\n    不要引用来自其他域名的内容。"

add_user_message(
    messages,
    f"""
    中国AI大模型，近期有哪些进展？{domain_hint}
    返回数据格式为数组每个数据为json格式，json格式如下：
    {{
        "url": "https://www.baidu.com",
        "content": "中国AI大模型领域近期取得了显著进展，呈现出技术创新加速、市场规模迅速扩大以及商业化应用深入发展的趋势。"
    }}
    """,
)
response = chat(messages, tools=[web_search_schema])
response

InternalServerError: Error code: 502

In [ ]:
# 解析响应内容，提取文本和链接，生成可点击的网页
import re
import json
from pathlib import Path

def parse_json_summaries(json_text):
    """从 JSON 文本中解析文献摘要"""
    try:
        # 提取 JSON 部分（去除 ```json 和 ```）
        json_match = re.search(r'```json\s*(.*?)\s*```', json_text, re.DOTALL)
        if json_match:
            json_str = json_match.group(1)
        else:
            json_str = json_text
        
        # 解析 JSON
        summaries = json.loads(json_str)
        return summaries
    except Exception as e:
        print(f"⚠️ JSON 解析错误: {e}")
        return []

def parse_links(citation_text):
    """从引用文本中解析链接，格式为 [1] [domain.com](url)"""
    links = {}
    # 匹配 [数字] [domain](url) 格式
    pattern = r'\[(\d+)\]\s*\[([^\]]+)\]\(([^)]+)\)'
    matches = re.findall(pattern, citation_text)
    for num, domain, url in matches:
        links[int(num)] = {
            'domain': domain,
            'url': url
        }
    return links

def generate_html_with_summaries(json_text, citation_text, output_file='output.html', allowed_domains=None):
    """生成包含文献摘要的 HTML 网页
    
    Args:
        json_text: JSON 格式的文献摘要文本
        citation_text: 引用链接文本
        output_file: 输出 HTML 文件路径
        allowed_domains: 允许的域名列表，为 None 时不过滤
    """
    summaries = parse_json_summaries(json_text)
    links = parse_links(citation_text)
    
    # 按 allowed_domains 过滤
    if allowed_domains:
        summaries = [s for s in summaries if any(d in s.get('url', '') for d in allowed_domains)]
        links = {n: l for n, l in links.items() if any(l['domain'] == d or l['domain'].endswith('.' + d) for d in allowed_domains)}
        print(f"🔍 域名过滤后: 摘要 {len(summaries)} 条, 链接 {len(links)} 条（允许: {allowed_domains}）")
    
    # 生成完整的 HTML
    html_content = f"""<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AI大模型进展报告</title>
    <style>
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Microsoft YaHei', sans-serif;
            line-height: 1.8;
            max-width: 1000px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        .container {{
            background-color: white;
            padding: 40px;
            border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h1, h2, h3 {{
            color: #333;
        }}
        a {{
            color: #0066cc;
            text-decoration: none;
            border-bottom: 1px solid #0066cc;
        }}
        a:hover {{
            color: #004499;
            border-bottom-color: #004499;
        }}
        .section {{
            margin-top: 40px;
            padding-top: 20px;
            border-top: 2px solid #eee;
        }}
        .citation-item {{
            margin: 20px 0;
            padding: 20px;
            background-color: #f9f9f9;
            border-left: 4px solid #0066cc;
            padding-left: 20px;
            border-radius: 4px;
        }}
        .citation-header {{
            margin-bottom: 12px;
            font-size: 16px;
        }}
        .citation-summary {{
            margin-top: 12px;
            padding: 12px;
            background-color: #ffffff;
            border-radius: 4px;
            color: #555;
            line-height: 1.6;
        }}
        .citation-number {{
            font-weight: bold;
            color: #0066cc;
            font-size: 18px;
        }}
        .citation-domain {{
            margin-left: 8px;
            font-size: 14px;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 中国AI大模型近期进展报告</h1>
        
        <div class="section">
            <h2>📝 文献摘要</h2>
"""
    
    # 展示所有摘要
    for i, item in enumerate(summaries, 1):
        url = item.get('url', '#')
        content = item.get('content', '暂无摘要')
        html_content += f"""            <div class="citation-item">
                <div class="citation-header">
                    <span class="citation-number">[{i}]</span>
                    <span class="citation-domain">
                        <a href="{url}" target="_blank">{url[:60]}...</a>
                    </span>
                </div>
                <div class="citation-summary">
                    {content}
                </div>
            </div>
"""
    
    # 展示所有引用链接
    html_content += """        </div>
        
        <div class="section">
            <h2>🔗 来源引文</h2>
"""
    for num in sorted(links.keys()):
        link = links[num]
        html_content += f"""            <div class="citation-item">
                <div class="citation-header">
                    <span class="citation-number">[{num}]</span>
                    <span class="citation-domain">
                        <a href="{link['url']}" target="_blank">{link['domain']}</a>
                    </span>
                </div>
            </div>
"""
    
    html_content += """        </div>
    </div>
</body>
</html>"""
    
    # 保存 HTML 文件
    output_path = Path(output_file)
    output_path.write_text(html_content, encoding='utf-8')
    
    print(f"✅ HTML 文件已生成: {output_path.absolute()}")
    print(f"📝 摘要数量: {len(summaries)} 条")
    print(f"🔗 引用链接数量: {len(links)} 条")
    return html_content

In [ ]:
response

Message(id='zwCPadPgBsac3LUPhPjP-QU', content=[TextBlock(citations=None, text='```json\n[\n  {\n    "url": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEBZC8Zo9JqBzqfAlx32xO4DW1nIfBaUaogAcWL_B75jT40c1efZrJz_5XB6gj4oFR6roQTa1nABrsKWewFUv10WfD6CbDBfyfE2Sy4FsKOqmXPk8AlnoLKQuaBAA==",\n    "content": "2025年1月，深度求索发布开源推理模型DeepSeek R1，其性能媲美OpenAI o1，训练成本仅约560万美元，迅速登顶全球应用商店榜首。该事件标志着中国大模型技术实现重大突破，重塑全球AI竞争格局，成为年度人工智能领域重要里程碑，彰显中国在AI大模型研发与应用的国际影响力。"\n  },\n  {\n    "url": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEBZC8Zo9JqBzqfAlx32xO4DW1nIfBaUaogAcWL_B75jT40c1efZrJz_5XB6gj4oFR6roQTa1nABrsKWewFUv10WfD6CbDBfyfE2Sy4FsKOqmXPk8AlnoLKQuaBAA==",\n    "content": "阿里云发布了通义千问旗舰版模型Qwen2.5-Max，该模型在多项公开主流模型评测基准上录得高分，全面超越了当时全球领先的开源MoE模型以及最大的开源稠密模型。同月，阿里云通义千问还开源了全新的视觉模型Qwen2.5-VL，推出3B、7B和72B三个尺寸版本，能够更准确地解析图像内容，并突破性地支持超1小时的视频理解。"\n  },\n  {\n    "url": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEBZC8Zo9JqBzqfAlx32xO4DW1nIfBaUaogAcWL_

In [ ]:
# 生成网页：将文献摘要与链接一一匹配，生成可点击的网页
if len(response.content) >= 2:
    json_text = response.content[0].text  # JSON 格式的文献摘要
    citation_text = response.content[1].text  # 引用链接
    
    # 从 web_search_schema 获取允许的域名
    allowed_domains = web_search_schema.get('allowed_domains', None)
    
    # 生成 HTML 网页（包含文献摘要），并过滤允许的域名
    html_content = generate_html_with_summaries(
        json_text, 
        citation_text, 
        'ai_model_progress.html',
        allowed_domains=allowed_domains
    )
    
    # 显示预览信息
    summaries = parse_json_summaries(json_text)
    links = parse_links(citation_text)
    print(f"📄 文献摘要数量: {len(summaries)} 个")
    print(f"🔗 引用链接数量: {len(links)} 个")
    if allowed_domains:
        print(f"✅ 允许的域名: {allowed_domains}")
    print(f"\n🌐 网页已生成，可以在浏览器中打开查看！")
else:
    print("⚠️ 响应内容不足，无法生成网页")

🔍 域名过滤后: 摘要 0 条, 链接 0 条（允许: ['x.com', 'infoq.cn', 'aibase.com']）
✅ HTML 文件已生成: /Users/canglong/Documents/deep-learning/building_with_claudeCode_api/ai_model_progress.html
📝 摘要数量: 0 条
🔗 引用链接数量: 0 条
📄 文献摘要数量: 15 个
🔗 引用链接数量: 8 个
✅ 允许的域名: ['x.com', 'infoq.cn', 'aibase.com']

🌐 网页已生成，可以在浏览器中打开查看！


# 在 Jupyter 中预览生成的网页
from IPython.display import HTML, display

if len(response.content) >= 2:
    json_text = response.content[0].text
    citation_text = response.content[1].text
    html_content = generate_html_with_summaries(json_text, citation_text, 'ai_model_progress.html')
    display(HTML(html_content))
else:
    print("⚠️ 响应内容不足，无法显示网页")